In [1]:
from pyscf import gto, scf, cc
import numpy as np
from jax import numpy as jnp
from jax import vmap, jvp, jit
import jax
from functools import partial

a = 2 # 2aB
nH = 2
atoms = ""
for i in range(nH):
    atoms += f"O {i*a:.5f} 0.00000 0.00000 \n"

mol = gto.M(atom=atoms, basis="6-31g", unit='B', spin=0, verbose=4)
mol.build()

mf = scf.RHF(mol)
mf.kernel()

mo = mf.stability()[0]
dm = mf.make_rdm1(mo,mf.mo_occ)
mf.kernel(dm0=dm)
mo = mf.stability()[0]
dm = mf.make_rdm1(mo,mf.mo_occ)
mf.kernel(dm0=dm)


nfrozen = 0
mycc = cc.CCSD(mf,frozen=nfrozen)
mycc.kernel()
# et = mycc.ccsd_t()
# print(mycc.e_tot + et)
mycc.energy()

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-14-generic', version='#14~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Thu Jan 15 15:52:10 UTC 2', machine='x86_64')  Threads 16
Python 3.11.14 (main, Oct 21 2025, 18:31:21) [GCC 11.2.0]
numpy 2.3.1  scipy 1.16.2  h5py 3.14.0
Date: Fri Feb 13 13:45:54 2026
PySCF version 2.11.0
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT HEAD (branch master) 3d1768f5e33b144b606c3d2c81c12ee54d794501

[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 2
[INPUT] num. electrons = 16
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = B
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 O      0.000000000000   0.000000000000   0.000000000000 AA    0.000000000000   0.000000000000   0.000000000000 Bohr 

np.float64(-0.2321035150390511)

In [2]:
mycc.energy(mycc.t1,mycc.t2*0)

np.float64(-4.1274411329342614e-05)

In [81]:
# example for PT2
options = {'n_eql': 3,
           'n_prop_steps': 50,
            'n_ene_blocks': 1,
            'n_sr_blocks': 5,
            'n_blocks': 50,
            'n_walkers': 100,
            'seed': 2,
            'walker_type': 'rhf',
            'trial': 'ccsd_pt2',
            'dt':0.005,
            'free_projection':False,
            'fp_abs': False,
            'group': False,
            'ad_mode':None,
            'use_gpu': False,
            }

from ad_afqmc.prop_unrestricted import prep
prep.prep_afqmc(mycc,options,chol_cut=1e-5)
# prop_unrestricted.run_afqmc(options,nproc=1)
option_file='options.bin'
import pickle
with open(option_file, 'wb') as f:
    pickle.dump(options, f)

#
# Preparing AFQMC calculation
# If you import pyscf cc modules and use MPI for AFQMC in the same script, finalize MPI before calling the AFQMC driver.
# Calculating Cholesky integrals
# Finished calculating Cholesky integrals
#
# Size of the correlation space:
# Number of electrons: (8, 8)
# Number of basis functions: 18
# Number of Cholesky vectors: 94
#


In [3]:
import numpy as np
from jax import random
from jax import numpy as jnp
from functools import partial
from ad_afqmc import config
from ad_afqmc.prop_unrestricted import sampling
import time

In [82]:
config.setup_jax()
MPI = config.setup_comm()
comm = MPI.COMM_WORLD
size = comm.Get_size()
rank = comm.Get_rank()

print = partial(print, flush=True)

ham_data, ham, prop, trial, wave_data, sampler, observable, options = (prep._prep_afqmc())

init_time = time.time()
comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

### initialize propagation
seed = options["seed"]
init_walkers = None
trial_rdm1 = trial.get_rdm1(wave_data)
if "rdm1" not in wave_data:
    wave_data["rdm1"] = trial_rdm1
ham_data = ham.build_measurement_intermediates(ham_data, trial, wave_data)
ham_data = ham.build_propagation_intermediates(ham_data, prop, trial, wave_data)
h0 = ham_data['h0']

prop_data = prop.init_prop_data(trial, wave_data, ham_data, init_walkers)
if jnp.abs(jnp.sum(prop_data["overlaps"])) < 1.0e-6:
    raise ValueError(
        "Initial overlaps are zero. Pass walkers with non-zero overlap."
    )
prop_data["key"] = random.PRNGKey(seed + rank)

prop_data["overlaps"] = trial.calc_overlap(prop_data["walkers"], wave_data)
prop_data["n_killed_walkers"] = 0

t1, t2, e0, e1 = trial.calc_energy_pt(prop_data["walkers"], ham_data, wave_data)
ept_sp = h0 + e0/t1 + e1/t1 - t2 * e0 / t1**2
ept = jnp.array(jnp.sum(ept_sp) / prop.n_walkers)
prop_data["e_estimate"] = ept
eci = trial.calc_energy(
    prop_data['walkers'], ham_data, wave_data)
prop_data["pop_control_ene_shift"] = prop_data["e_estimate"]

print(ept)
print(ept-mycc.e_tot)

# Hostname: sharmagroup-rn
# System Type: Linux
# Machine Type: x86_64
# Processor: x86_64
# Hostname: sharmagroup-rn
# System Type: Linux
# Machine Type: x86_64
# Processor: x86_64
# Number of MPI ranks: 1
#
# norb: 18
# nelec: (8, 8)
#
# n_eql: 3
# n_prop_steps: 50
# n_ene_blocks: 1
# n_sr_blocks: 5
# n_blocks: 50
# n_walkers: 100
# seed: 2
# walker_type: rhf
# trial: ccsd_pt2
# dt: 0.005
# free_projection: False
# fp_abs: False
# group: False
# use_gpu: False
# n_exp_terms: 6
# symmetry: False
# save_walkers: False
# ene0: 0.0
# n_batch: 1
# maxError: 0.001
#
-149.65038719037096
5.8141753243035055e-06


In [5]:
def hs_op_yann(self, wave_data: dict):
    # adapted from Yann

    nO = self.nelec[0]
    nV = self.norb - nO
    nex = nO * nV

    t2 = wave_data['t2']

    assert t2.shape == (nO, nV, nO, nV)

    # T2 = LL^T
    t2 = jnp.einsum("iajb->aibj", t2)
    # assert t2.shape == (nO, nV, nO, nV)
    
    t2 = t2.reshape(nex, nex)
    e_val, e_vec = jnp.linalg.eigh(t2)
    L = e_vec @ jnp.diag(np.sqrt(e_val + 0.0j))
    assert abs(jnp.linalg.norm(t2 - L @ L.T)) < 1e-12

    # Summation on the left
    L = L.T.reshape(nex, nV, nO)

    # wave_data["T2_L"] = L

    return L, e_val

# @partial(jax.jit, static_argnums=(0, 2))
def get_stocc_yann(self, wave_data: dict, nslater: int):
    nO = self.nelec[0]
    nV = self.norb - nO
    n = nO + nV
    nex = nO * nV

    t1 = wave_data["t1"]

    C_occ, C_vir = jnp.split(wave_data["mo_coeff_full"], [nO], axis=1)

    # e^T1
    e_t1 = t1.T + 0.0j
    ops = jnp.array([e_t1] * nslater)

    # L = wave_data["T2_L"]
    L, e_val = hs_op_yann(self, wave_data)

    wave_data["key"], subkey = random.split(wave_data["key"])
    fields = random.normal(
        subkey,
        shape=(
            nslater,
            L.shape[0],
        ),
    )

    # e^{t1+x*tau2}
    # t1s = jnp.array([t1 + 0.0j] * nslater)
    ops = ops + jnp.einsum("wg,gai->wai", fields, L)

    # Initial slater determinants
    rdm1 = self.get_rdm1(wave_data)
    natorbs_up = jnp.linalg.eigh(rdm1[0])[1][:, ::-1][:, : self.nelec[0]]

    stocc = jnp.array([natorbs_up + 0.0j] * nslater)
    identity = jnp.array([np.identity(n) + 0.0j] * nslater)

    # e^{T1+T2} \ket{\phi}
    stocc = (
        identity + jnp.einsum("pa,wai,iq -> wpq", C_vir, ops, C_occ.T)
    ) @ stocc
    stocc = jnp.array(stocc)

    return stocc

def decompose_t2(self, wave_data: dict):
    # adapted from Yann

    nO = self.nelec[0]
    nV = self.norb - nO
    nex = nO * nV

    t2 = wave_data['t2']

    # assert t2.shape == (nO, nO, nV, nV)

    # T2 = LL^T
    # t2 = jnp.einsum("iajb->aibj", t2)
    assert t2.shape == (nO, nV, nO, nV)
    
    t2 = t2.reshape(nex, nex)
    e_val, e_vec = jnp.linalg.eigh(t2)
    L = e_vec @ jnp.diag(np.sqrt(e_val + 0.0j))
    assert abs(jnp.linalg.norm(t2 - L @ L.T)) < 1e-12

    # Summation on the left
    L = L.T.reshape(nex, nO, nV)

    # wave_data["T2_L"] = L

    return L, e_val

# @partial(jax.jit, static_argnums=(0, 2))
def get_stocc_my(self, wave_data: dict, nslater: int):
    nO = self.nelec[0]
    t1 = wave_data["t1"]

    # L, e_val = hs_op_yann(self, wave_data)
    # L = L.transpose(0,2,1)
    L, e_val = decompose_t2(self, wave_data)

    wave_data["key"], subkey = random.split(wave_data["key"])
    fields = random.normal(
        subkey,
        shape=(
            nslater,
            L.shape[0],
        ),
    )

    # e^{t1+x*tau2}
    t1s = jnp.array([t1 + 0.0j] * nslater)
    taus = t1s + jnp.einsum("wg,gia->wia", fields, L)

    from jax import scipy as jsp
    def _exp_tau(tau, sd):
        # tau_full = jnp.zeros((self.norb, self.norb),dtype=jnp.complex128)
        tau_full = jnp.eye(self.norb,dtype=jnp.complex128)
        exp_tau = tau_full.at[:nO, nO:].set(tau)
        # exp_tau = jsp.linalg.expm(tau_full)
        return exp_tau.T @ sd

    # Initial slater determinants
    init_sd = jnp.array([jnp.eye(self.norb)[:,:nO] + 0.0j] * nslater)
    stocc = vmap(_exp_tau)(taus, init_sd)

    return stocc


def get_stocc_imp1(self, wave_data: dict, nslater: int):
    # importance sampling 1 p(x) = exp(-x|lamda-lamba_max|)
    nO = self.nelec[0]
    t1 = wave_data["t1"]

    L, e_val = hs_op(self, wave_data)
    # shift = jnp.array([jnp.abs(e_val - e_val.max())] * nslater)
    shift = -jnp.array([jnp.abs(e_val)] * nslater)

    wave_data["key"], subkey = random.split(wave_data["key"])
    fields = random.normal(
        subkey,
        shape=(
            nslater,
            L.shape[0],
        ),
    )

    # e^{t1+x*tau2}
    t1s = jnp.array([t1 + 0.0j] * nslater)
    fields = fields + shift
    taus = t1s + jnp.einsum("wg,gia->wia", fields, L)

    field_shift = jnp.einsum("wg,wg->w", fields, shift)
    shfit_shift = jnp.einsum("wg,wg->w", shift, shift)
    weight = jnp.exp(-field_shift + shfit_shift/2)

    from jax import scipy as jsp
    def _exp_tau(tau, sd):
        # tau_full = jnp.zeros((self.norb, self.norb),dtype=jnp.complex128)
        tau_full = jnp.eye(self.norb,dtype=jnp.complex128)
        # tau_full = tau_full.at[:nO, nO:].set(tau)
        exp_tau = tau_full.at[:nO, nO:].set(tau)
        # exp_tau = jsp.linalg.expm(tau_full)
        return exp_tau.T @ sd

    # Initial slater determinants
    init_sd = jnp.array([jnp.eye(self.norb)[:,:nO] + 0.0j] * nslater)
    stocc = vmap(_exp_tau)(taus, init_sd)

    return stocc, weight

In [ ]:
# myt, mye = hs_op(trial, wave_data)
# yat, yae = hs_op_yann(trial, wave_data)

In [ ]:
# myt2_rec = jnp.einsum('gia,gjb->iajb',myt,myt)
# jnp.abs(wave_data["t2"] - myt2_rec).max()

Array(1.11022302e-16, dtype=float64)

In [ ]:
# jnp.abs(myt - yat.transpose(0,2,1)).max()

Array(0.21153878, dtype=float64)

In [ ]:
# yat2_rec = jnp.einsum('gai,gbj->aibj',yat,yat)
# jnp.abs(wave_data["t2"] - yat2_rec.transpose(1,0,3,2)).max()

Array(1.12482328e-16, dtype=float64)

In [ ]:
# no = trial.nelec[0]
# nv = trial.norb - no
# myt = myt.reshape(myt.shape[0],no,nv)
# myt = myt.transpose(0,2,1)
# yat = yat.reshape(yat.shape[0],nv,no)

In [ ]:
# print(jnp.abs(myt-yat).max())

1.3687688811150533


In [13]:
# wave_data["key"] = prop_data["key"]
stocc = trial.get_stocc(wave_data, prop_data, nslater=1000)
e_stocc = trial.calc_energy(stocc, ham_data, wave_data)
olp_stocc = trial.calc_overlap(stocc, wave_data)
# print(e_stocc)
# print(olp_stocc)
e_cc = jnp.sum(olp_stocc * e_stocc) / jnp.sum(olp_stocc)
print(e_cc-mycc.e_tot)

(-0.0033278458575409786+0.00011491240615062751j)


In [ ]:
stocc

In [ ]:
import opt_einsum as oe

@jit
def get_green_slater(trial_slater: jax.Array, walker: jax.Array) -> jax.Array:
    
    gf = (
        walker @ (
            jnp.linalg.inv(trial_slater.T.conj() @ walker)
                ) @ trial_slater.T.conj()
        ).T
    
    return gf

@partial(jit, static_argnums=0)
def get_energy_slater(trial, slater: jax.Array, walker: jax.Array, ham_data: dict) -> jax.Array:
    norb = trial.norb

    h0, chol = ham_data["h0"], ham_data["chol"]
    h1 = (ham_data["h1"][0] + ham_data["h1"][1]) / 2.0
    chol = chol.reshape(-1,norb,norb)

    green = get_green_slater(slater, walker)
    hg = oe.contract("pq,pq->", h1, green, backend="jax")
    e1 = 2 * hg
    lg = oe.contract("gpr,qr->gpq", chol, green, backend="jax")
    e2_1 = 2 * jnp.sum(oe.contract('gpp->g', lg, backend="jax")**2)
    e2_2 = oe.contract('gpq,gqp->',lg,lg, backend="jax")
    e2 = e2_1 - e2_2

    return h0 + e1 + e2

@jit
def get_overlap_slater(slater: jax.Array, walker: jax.Array) -> jax.Array:
    return jnp.linalg.det(slater.T.conj() @ walker) ** 2

In [34]:
ham_data["h1"].shape

(2, 18, 18)

In [20]:
walker = prop_data['walkers'][0]
slater = jnp.eye(trial.norb)[:,:trial.nelec[0]]
green = get_green_slater(slater, walker)
print(green.shape)

(18, 18)


In [29]:
green.diagonal()

Array([1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j,
       0.+0.j, 0.+0.j], dtype=complex128)

In [23]:
ham_data["chol"].reshape(-1,trial.norb,trial.norb).shape

(94, 18, 18)

In [21]:
trial.nelec[0]

8

In [26]:
wave_data["mo_coeff"]

Array([[1., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float64)

In [62]:
walker = np.random.rand(*walker.shape)
e1 = get_energy_slater(trial, slater, walker, ham_data)
print(e1-mf.e_tot)
e2 = trial._calc_energy_restricted(walker, ham_data, wave_data)
print(e2-mf.e_tot)

2.8081512087398437
2.808151208739787


In [97]:
from jax import lax

@partial(jit, static_argnums=0)
def get_energy_slaters_one_walker(
    trial, 
    slaters: jax.Array,
    walker: jax.Array,
    ham_data: dict
    ):
    """
    slaters: (N, norb, nocc)
    walker:  (norb, nocc)

    returns: (N,) energies
    """

    def scan_slaters(carry, slater):
        # carry is unused; we keep it for scan API
        energy = get_energy_slater(trial, slater, walker, ham_data)
        return carry, energy

    # Initial dummy carry (None not allowed)
    init_carry = 0.0

    _, energies = lax.scan(scan_slaters, init_carry, slaters)

    return energies

@jit
def get_overlap_slaters_one_walker(
    slaters: jax.Array,
    walker: jax.Array,
    ):
    """
    slaters: (N, norb, nocc)
    walker:  (norb, nocc)

    returns: (N,) energies
    """

    def scan_slaters(carry, slater):
        # carry is unused; we keep it for scan API
        overlap = get_overlap_slater(slater, walker)
        return carry, overlap

    # Initial dummy carry (None not allowed)
    init_carry = 0.0

    _, overlaps = lax.scan(scan_slaters, init_carry, slaters)

    return overlaps

In [ ]:
# def get_energy_slaters_one_walker(trial, slaters, walker, ham_data):
#     # slaters: (M, norb, norb)
#     # walker: (norb, nocc)
#     def warp_get_energy_slater(trial, slater, walker, ham_data):
#         return get_energy_slater(trial, slater, walker, ham_data)
    
    
#     return vmap(
#         lambda slater: get_energy_slater(trial, slater, walker, ham_data)
#     )(slaters)

# def get_overlap_slaters_one_walker(slaters, walker):
#     # slaters: (M, norb, norb)
#     # walker: (norb, nocc)
#     return vmap(
#         lambda slater: get_overlap_slater(slater, walker)
#     )(slaters)

In [98]:
walker = prop_data['walkers'][0]
e_stocc_walker = get_energy_slaters_one_walker(trial, stocc, walker, ham_data)
o_stocc_walker = get_overlap_slaters_one_walker(stocc, walker)

In [99]:
jnp.sum(o_stocc_walker * e_stocc_walker) / jnp.sum(o_stocc_walker) - mycc.e_tot

Array(-0.00332785-0.00011491j, dtype=complex128)

In [112]:
def get_eloc_oloc_stocc(
    self, walker: jax.Array, ham_data: dict, wave_data: dict
) -> jax.Array:
    slaters = wave_data['stocc']
    energies = get_energy_slaters_one_walker(self, slaters, walker, ham_data)
    overlaps = get_overlap_slaters_one_walker(slaters, walker) / slaters.shape[0]
    oloc = jnp.sum(overlaps)
    eloc = jnp.sum(overlaps * energies) / oloc
    return (eloc, oloc) 

In [113]:
wave_data['stocc'] = stocc
(eloc, oloc) = get_eloc_oloc_stocc(trial, walker, ham_data, wave_data)

In [114]:
eloc - mycc.e_tot

Array(-0.00332785-0.00011491j, dtype=complex128)

In [116]:
def calc_energy_overlap_stocc(
        self, walkers: jax.Array, ham_data: jax.Array, wave_data: dict
        ):

    (energies, overlaps) =  vmap(
        lambda walker: get_eloc_oloc_stocc(self, walker, ham_data, wave_data
        ))(walkers)
    
    return (energies, overlaps)

In [117]:
(energies, overlaps) = calc_energy_overlap_stocc(
        trial, prop_data['walkers'], ham_data, wave_data
    )

In [120]:
print(energies.shape)
print(overlaps.shape)

(100,)
(100,)


In [121]:
overlaps

Array([1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j], dtype=complex128)

In [ ]:
# def energy_matrix(trial, slaters, walkers, ham_data):
#     # slaters: (M, norb, nocc)
#     # walkers: (N, norb, nocc)
#     # return energies: (N, M)
#     return vmap(
#         lambda walker: get_energy_slaters_one_walker(
#             trial, slaters, walker, ham_data
#         )
#     )(walkers)

# def overlap_matrix(slaters, walkers):
#     # slaters: (M, norb, nocc)
#     # walkers: (N, norb, nocc)
#     # return overlaps: (N, M)
#     return vmap(
#         lambda walker: get_overlap_slaters_one_walker(
#             slaters, walker
#         )
#     )(walkers)

In [ ]:
# e_matric = energy_matrix(trial, stocc, prop_data['walkers'], ham_data)
# o_matric = overlap_matrix(stocc, prop_data['walkers'])